# 02 — Held-out evaluation and dependence-aware statistics


**Purpose.** Evaluate only the predictions generated by Notebook 01. This notebook computes site-level and site-macro metrics, compares the new run with a compact paper reference table, and performs a moving-block bootstrap sanity check. It does not retrain BEDOL.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT/"src"))
from bedol_repro import *

OUT = ROOT/"reproduction_outputs"
CONFIG_DIR, DATA_DIR = ROOT/"configs", ROOT/"data"
pred = pd.read_csv(OUT/"01_bedol_test_predictions.csv", parse_dates=["origin","target_time"])
ref = pd.read_csv(ROOT/"reference"/"paper_reference_metrics.csv")
manifest, frames = load_public_data(DATA_DIR, CONFIG_DIR/"site_manifest.csv")
splits = pd.read_csv(CONFIG_DIR/"chronological_splits.csv")
proc, meta, site_to_idx, hist_features, future_features = prepare_frames(frames,splits)


In [ ]:
# Primary metrics use recommended_for_evaluation=1.
site_metrics, macro_metrics = summarize_predictions(pred, primary_only=True)
site_metrics.to_csv(OUT/"02_site_metrics.csv", index=False)
macro_metrics.to_csv(OUT/"02_macro_metrics.csv", index=False)
display(site_metrics)
display(macro_metrics)
display(ref)

paper_nrmse = float(ref.loc[ref.quantity.eq("BEDOL_macro_nRMSE"),"value"].iloc[0])
fresh_nrmse = float(macro_metrics.nRMSE.iloc[0])
print(f"Fresh macro nRMSE: {fresh_nrmse:.6f}")
print(f"Compact paper reference: {paper_nrmse:.6f}")
print("Small differences across hardware/software are expected; do not tune on the test year to close the gap.")


In [ ]:
# 24-hour seasonal-naive comparator on the identical evaluated target points.
naive = pred[["site_id","origin","horizon","target_time","y_true","capacity","eval_flag"]].copy()
naive_pred = []
for row in naive.itertuples(index=False):
    raw = frames[row.site_id].set_index("ds")[TARGET]
    source_time = row.target_time - pd.Timedelta(hours=24)
    naive_pred.append(float(raw.get(source_time, np.nan)))
naive["pred"] = naive_pred
naive = naive.dropna(subset=["pred"])

_, naive_macro = summarize_predictions(naive, primary_only=True)
display(naive_macro)


In [ ]:
# Site-stratified moving-block bootstrap over forecast origins.
BOOT_REPS = 5000
obs, lo, hi = moving_block_bootstrap(
    origin_loss(pred), origin_loss(naive),
    block=28, reps=BOOT_REPS, seed=20260929
)
stats = pd.DataFrame([{
    "comparison":"BEDOL minus seasonal-naive",
    "mean_normalized_squared_loss_diff":obs,
    "ci_lo":lo, "ci_hi":hi,
    "block_origins":28, "bootstrap_reps":BOOT_REPS
}])
stats.to_csv(OUT/"02_block_bootstrap.csv",index=False)
display(stats)
print("Negative loss difference favors BEDOL.")
